# 🧬 Holo-GNN · Full-Scale Training on Vertex AI (NVIDIA L4)

Single-file, self-contained notebook. Execute **Runtime → Run all** to run the complete pipeline:
data download → audit → model build → training → checkpoint.

| Setting | Value |
|---|---|
| GPU | NVIDIA L4 (24 GB VRAM) |
| vCPUs | 8 data-loader workers |
| Physical Batch Size | 64 |
| Gradient Accumulation | 1 (native — no accumulation overhead) |
| Effective Batch Size | **64** |

---
## Cell 1 — Data Pipeline (GCS → Local)

In [ ]:
# ── 1a. Copy dataset archive from Google Cloud Storage ───────────────────────
!gcloud storage cp gs://modelrundatas/data.zip .

# ── 1b. Unzip quietly (suppress per-file output) ─────────────────────────────
!unzip -q data.zip

print("✅ data.zip fetched from GCS and extracted successfully.")

---
## Cell 2 — Dataset Classes
*Inlined from `src/dataset.py`*

In [ ]:
# ── Shared imports ────────────────────────────────────────────────────────────
import os, time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from tqdm.notebook import tqdm
from transformers import EsmTokenizer, EsmModel
from Bio.Seq import Seq

# Suppress duplicate-lib warnings (harmless on Linux)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ── PyG (optional — enables GATConv message passing) ────────────────────────
try:
    from torch_geometric.nn import GATConv
    print("✅ torch_geometric.nn.GATConv available — GNN layers ENABLED.")
except ImportError:
    GATConv = None
    print("⚠️  torch_geometric not found — falling back to linear pooling.")

# ── Device report ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — running on CPU.")
print(f"   Device: {device}")

# ─────────────────────────────────────────────────────────────────────────────
# Graph builder utilities (inlined from src/utils/graph_builder.py)
# ─────────────────────────────────────────────────────────────────────────────

def build_graph_from_attention(attention_map, threshold=0.1):
    """
    Constructs a sparse edge_index from an ESM-2 attention map.
    Symmetrises the map and keeps only edges above `threshold`.
    """
    adj = attention_map + attention_map.t()
    rows, cols = torch.where(adj > threshold)
    mask = rows != cols          # Remove self-loops
    return torch.stack([rows[mask], cols[mask]], dim=0)


def simple_linear_graph(seq_len):
    """
    Fallback graph: connects residue i → i+1 (peptide-backbone chain).
    Bi-directional and compatible with arbitrary batch sizes.
    """
    src = torch.arange(0, seq_len - 1)
    tgt = torch.arange(1, seq_len)
    return torch.stack([torch.cat([src, tgt]), torch.cat([tgt, src])], dim=0)


# ─────────────────────────────────────────────────────────────────────────────
# Dataset classes (inlined from src/dataset.py)
# ─────────────────────────────────────────────────────────────────────────────

class MassIVEKBDataset(Dataset):
    """
    Loader for MassIVE-KB spectral library (.sptxt files).
    Extracts peptide sequences for pre-training.
    Labels are dummy (0.0) — RetentionTime tags are absent in raw files.
    """

    def __init__(self, data_dir, max_length=100):
        self.tokenizer  = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
        self.data       = []
        self.max_length = max_length

        print(f"Scanning {data_dir} for .sptxt files...")
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.endswith(".sptxt"):
                    self._parse_sptxt(os.path.join(root, f))
        print(f"Loaded {len(self.data):,} sequences from MassIVE-KB.")

    def _parse_sptxt(self, filepath):
        with open(filepath, "r") as f:
            for line in f:
                if line.startswith("Name:"):
                    try:
                        raw   = line.strip().split(" ")[1]
                        seq   = "".join(c for c in raw.split("/")[0] if c.isalpha())
                        if seq:
                            self.data.append({"seq": seq, "label": 0.0})
                    except IndexError:
                        continue

    def __len__(self):  return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["seq"], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(item["label"], dtype=torch.float),
        }


class MegaScaleDataset(Dataset):
    """
    Loader for the Mega-scale cDNA stability dataset.
    Translates 'dna_seq' → protein sequence on-the-fly via Biopython.
    Regression target: 'deltaG' (thermodynamic stability, kcal/mol).
    """

    def __init__(self, csv_path, max_length=100):
        self.tokenizer  = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
        self.max_length = max_length

        print(f"Loading Mega-scale data from {csv_path}...")
        df = pd.read_csv(csv_path)
        self.df = df.dropna(subset=["dna_seq", "deltaG"]).reset_index(drop=True)
        print(f"Loaded {len(self.df):,} valid samples (DNA→Protein on-the-fly).")

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        protein_seq = str(Seq(row["dna_seq"]).translate(to_stop=True))
        label       = float(row["deltaG"])
        enc = self.tokenizer(
            protein_seq, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(label, dtype=torch.float),
        }


print("\n✅ Cell 2 complete — dataset classes and graph utilities defined.")

---
## Cell 3 — Model Architecture
*Inlined from `src/backbone.py`, `src/heads.py`, and `src/full_model.py`*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Prediction heads  (inlined from src/heads.py)
# ─────────────────────────────────────────────────────────────────────────────

class ProteomicsHead(nn.Module):
    """Retention-time regressor for MassIVE-KB pre-training."""
    def __init__(self, input_dim=512):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, 1)
        )
    def forward(self, graph_emb): return self.regressor(graph_emb)


class SiameseStabilityHead(nn.Module):
    """
    Predicts ΔΔG from the difference vector between wild-type
    and mutant graph embeddings (anti-symmetric Siamese design).
    """
    def __init__(self, input_dim=512):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, 1)
        )
    def forward(self, emb_wt, emb_mut): return self.mlp(emb_mut - emb_wt)


class EnsembleIDRHead(nn.Module):
    """
    Predicts μ and σ of the radius of gyration for disordered regions
    (ClinVar / IDR regression task).
    """
    def __init__(self, input_dim=512):
        super().__init__()
        self.mu_layer    = nn.Linear(input_dim, 1)
        self.sigma_layer = nn.Linear(input_dim, 1)
        self.softplus    = nn.Softplus()    # Guarantees σ > 0
    def forward(self, graph_emb):
        return self.mu_layer(graph_emb), self.softplus(self.sigma_layer(graph_emb))


# ─────────────────────────────────────────────────────────────────────────────
# Backbone  (inlined from src/backbone.py)
# ─────────────────────────────────────────────────────────────────────────────

class HoloGNNBackbone(nn.Module):
    """
    Dual-track backbone:
      Track 1 — ESM-2 (t6 8M) language model for sequence embeddings.
      Track 2 — GATConv message passing on the residue contact graph.
    Falls back to mean-pooled ESM-2 embeddings if torch_geometric is absent.
    """

    def __init__(self, hidden_dim=320, output_dim=320):
        super().__init__()
        self.esm_model = EsmModel.from_pretrained(
            "facebook/esm2_t6_8M_UR50D", output_attentions=True
        )
        if GATConv is not None:
            self.gat1 = GATConv(hidden_dim, hidden_dim, heads=4, concat=False)
            self.gat2 = GATConv(hidden_dim, output_dim, heads=4, concat=False)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, edge_index=None):
        # Step 1 — ESM-2 sequence embeddings
        out  = self.esm_model(input_ids=input_ids, attention_mask=attention_mask)
        node = out.last_hidden_state                       # (B, L, H)
        B, L, H = node.shape
        x = node.view(-1, H)                              # (B*L, H)

        # Step 2 — Build linear backbone graph if no edges supplied
        if edge_index is None and GATConv is not None:
            edge_index = simple_linear_graph(L).to(input_ids.device)

        # Step 3 — GAT message passing
        if edge_index is not None and GATConv is not None:
            x = self.relu(self.gat1(x, edge_index))
            x = self.gat2(x, edge_index)

        # Step 4 — Mean pooling → graph-level embedding
        x_reshaped = x.view(B, L, -1)
        graph_emb  = torch.mean(x_reshaped, dim=1)       # (B, H)
        return x_reshaped, graph_emb


# ─────────────────────────────────────────────────────────────────────────────
# Full model  (inlined from src/full_model.py)
# ─────────────────────────────────────────────────────────────────────────────

class HoloGNN(nn.Module):
    """
    HoloGNN — multi-task protein structure & function predictor.

    Tasks
    -----
    'proteomics' : Retention-time regression (MassIVE-KB)
    'idr'        : ΔG stability regression (MegaScale cDNA)
    default      : Returns raw graph embedding
    """

    def __init__(self, hidden_dim=320, output_dim=320):
        super().__init__()
        self.backbone        = HoloGNNBackbone(hidden_dim=hidden_dim, output_dim=output_dim)
        self.proteomics_head = ProteomicsHead(input_dim=output_dim)
        self.siamese_head    = SiameseStabilityHead(input_dim=output_dim)
        self.idr_head        = EnsembleIDRHead(input_dim=output_dim)

    def forward(self, data, task="proteomics"):
        # data must expose: .input_ids  .mask  .edge_index
        _, graph_emb = self.backbone(data.input_ids, data.mask, data.edge_index)

        if task == "proteomics":
            return self.proteomics_head(graph_emb)
        if task == "idr":
            return self.siamese_head.mlp(graph_emb)   # Scalar ΔG regression
        return graph_emb


# ── Quick parameter count ────────────────────────────────────────────────────
_tmp       = HoloGNN()
_total     = sum(p.numel() for p in _tmp.parameters())
_trainable = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
del _tmp

print("✅ Cell 3 complete — model architecture defined.")
print(f"   Total parameters    : {_total:,}")
print(f"   Trainable parameters: {_trainable:,}")

---
## Cell 4 — Hardware Hyperparameters & Data Preparation
*Optimised for NVIDIA L4 (24 GB VRAM) + 8 vCPU workers*

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# L4 HARDWARE SETTINGS  ←  Edit only this block
# ═════════════════════════════════════════════════════════════════════════════
BATCH_SIZE         = 64    # Saturates L4's 24 GB VRAM efficiently
ACCUMULATION_STEPS = 1     # Native batch — no accumulation overhead needed
NUM_WORKERS        = 8     # Matches available vCPUs for non-blocking IO

# ── Training settings ────────────────────────────────────────────────────────
LEARNING_RATE  = 1e-4
EPOCHS         = 5
MAX_SAMPLES    = 100_000   # Subset of MegaScale (speed ↔ accuracy balance)
VAL_SPLIT      = 0.10      # 90 % train / 10 % validation
MAX_SEQ_LENGTH = 100

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH = (
    "data/mega_scale_cdna/Processed_K50_dG_datasets/"
    "Processed_K50_dG_datasets/Tsuboyama2023_Dataset1_20230416.csv"
)
CHECKPOINT_PATH = "holognn_stability_final.pth"

# ── Summary ──────────────────────────────────────────────────────────────────
print("L4 Configuration loaded.")
print(f"  Batch size (physical) : {BATCH_SIZE}")
print(f"  Accumulation steps    : {ACCUMULATION_STEPS}")
print(f"  Effective batch size  : {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"  DataLoader workers    : {NUM_WORKERS}")
print(f"  Epochs                : {EPOCHS}")
print(f"  Max samples           : {MAX_SAMPLES:,}")

# ── Build DataLoaders ─────────────────────────────────────────────────────────
print("\n--- Preparing MegaScale Dataset ---")
full_dataset   = MegaScaleDataset(DATA_PATH, max_length=MAX_SEQ_LENGTH)

n_samples      = min(MAX_SAMPLES, len(full_dataset))
active_dataset = Subset(full_dataset, range(n_samples))

train_size     = int((1.0 - VAL_SPLIT) * n_samples)
val_size       = n_samples - train_size
train_ds, val_ds = random_split(active_dataset, [train_size, val_size])

# pin_memory + persistent_workers → maximum GPU-feed throughput on L4
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

print(f"\n✅ Cell 4 complete — data loaders ready.")
print(f"   Training samples   : {len(train_ds):,}")
print(f"   Validation samples : {len(val_ds):,}")
print(f"   Batches per epoch  : {len(train_loader):,}")

---
## Cell 5 — Data Auditor

Scans the active dataset **before training begins** and raises an error if any
critical issues are found, preventing silent failures mid-epoch.

Checks performed:
- NaN / Inf values in `input_ids`, `attention_mask`, and `label`
- NaN / Inf `deltaG` labels in the raw dataframe
- Translated protein sequence length distribution
- Sequences exceeding `MAX_SEQ_LENGTH` (will be silently truncated by the tokenizer)
- Empty sequences after DNA→Protein translation

In [ ]:
import math
from collections import Counter

def audit_dataset(dataset, name="Dataset", n_probe=2048):
    """
    Automated data quality auditor.

    Parameters
    ----------
    dataset : torch.utils.data.Dataset
        The dataset to audit (can be a Subset).
    name : str
        Label to print in the report header.
    n_probe : int
        Maximum number of samples to probe (avoids long audit on huge datasets).
    """
    SEP = "=" * 62
    sep = "-" * 62

    print(f"\n{SEP}")
    print(f"  🔍 DATA AUDITOR — {name}")
    print(SEP)

    issues = []
    n_probe  = min(n_probe, len(dataset))

    # ── 1. Raw DataFrame checks (MegaScaleDataset only) ──────────────────────
    if hasattr(dataset, "dataset") and hasattr(dataset.dataset, "df"):
        raw_df = dataset.dataset.df
    elif hasattr(dataset, "df"):
        raw_df = dataset.df
    else:
        raw_df = None

    if raw_df is not None:
        nan_dg  = raw_df["deltaG"].isna().sum()
        inf_dg  = (~raw_df["deltaG"].apply(lambda x: math.isfinite(x) if isinstance(x, float) else True)).sum()
        nan_dna = raw_df["dna_seq"].isna().sum()

        print(f"\n  [1] Raw DataFrame ({len(raw_df):,} rows)")
        print(sep)
        print(f"  NaN  in deltaG : {nan_dg}   {'⚠️ ISSUE' if nan_dg  else '✅ OK'}")
        print(f"  Inf  in deltaG : {inf_dg}   {'⚠️ ISSUE' if inf_dg  else '✅ OK'}")
        print(f"  NaN  in dna_seq: {nan_dna}  {'⚠️ ISSUE' if nan_dna else '✅ OK'}")

        if nan_dg:  issues.append(f"{nan_dg} NaN delta-G values")
        if inf_dg:  issues.append(f"{inf_dg} Inf delta-G values")
        if nan_dna: issues.append(f"{nan_dna} NaN dna_seq rows")

        # Delta-G statistics
        dg = raw_df["deltaG"].dropna()
        print(f"\n  deltaG stats   : min={dg.min():.2f}  max={dg.max():.2f}  "
              f"mean={dg.mean():.2f}  std={dg.std():.2f}")

    # ── 2. Tensor / tokenised sample checks ──────────────────────────────────
    print(f"\n  [2] Tokenised Sample Probe (first {n_probe:,} samples)")
    print(sep)

    nan_ids  = nan_mask = nan_labels = 0
    empty_seqs        = 0
    len_hist          = Counter()
    over_max          = 0

    for i in range(n_probe):
        sample = dataset[i]
        ids    = sample["input_ids"]
        mask   = sample["attention_mask"]
        label  = sample["label"]

        # NaN / Inf checks on tensors
        if torch.isnan(ids.float()).any() or torch.isinf(ids.float()).any():
            nan_ids += 1
        if torch.isnan(mask.float()).any() or torch.isinf(mask.float()).any():
            nan_mask += 1
        if torch.isnan(label).any() or torch.isinf(label).any():
            nan_labels += 1

        # Effective sequence length = number of non-padding tokens
        eff_len = int(mask.sum().item())
        bucket  = (eff_len // 10) * 10        # Round to nearest 10
        len_hist[bucket] += 1

        if eff_len == 0:
            empty_seqs += 1
        if eff_len >= MAX_SEQ_LENGTH:         # Will be truncated
            over_max += 1

    print(f"  NaN/Inf in input_ids      : {nan_ids}    {'⚠️ ISSUE' if nan_ids  else '✅ OK'}")
    print(f"  NaN/Inf in attention_mask : {nan_mask}   {'⚠️ ISSUE' if nan_mask else '✅ OK'}")
    print(f"  NaN/Inf in labels         : {nan_labels} {'⚠️ ISSUE' if nan_labels else '✅ OK'}")
    print(f"  Empty sequences (len=0)   : {empty_seqs} {'⚠️ ISSUE' if empty_seqs else '✅ OK'}")
    print(f"  Sequences ≥ MAX_SEQ_LEN   : {over_max}  "
          f"({'will be truncated' if over_max else '✅ None — all sequences fit'})")

    if nan_ids:    issues.append(f"{nan_ids} samples with NaN/Inf input_ids")
    if nan_labels: issues.append(f"{nan_labels} samples with NaN/Inf labels")
    if empty_seqs: issues.append(f"{empty_seqs} empty sequences (len=0)")

    # ── 3. Sequence length histogram ─────────────────────────────────────────
    print(f"\n  [3] Effective Sequence Length Distribution")
    print(sep)
    max_count = max(len_hist.values()) if len_hist else 1
    for bucket in sorted(len_hist):
        count  = len_hist[bucket]
        bar    = "█" * int(30 * count / max_count)
        print(f"  {bucket:>4}–{bucket+9:<4} | {bar:<30} {count}")

    # ── 4. Final verdict ──────────────────────────────────────────────────────
    print(f"\n{SEP}")
    if issues:
        print("  ❌ AUDIT FAILED — issues detected:")
        for iss in issues:
            print(f"     • {iss}")
        print(SEP)
        raise RuntimeError(
            "Data audit failed. Fix the issues listed above before training."
        )
    else:
        print("  ✅ AUDIT PASSED — dataset is clean. Safe to proceed.")
    print(SEP)


# ── Run the auditor on the training subset ───────────────────────────────────
audit_dataset(train_ds, name="MegaScale train split", n_probe=2048)

---
## Cell 6 — Model, Optimiser & Scheduler Initialisation

In [ ]:
print(f"Initialising HoloGNN on {device}...")

model     = HoloGNN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# OneCycleLR — warms up then anneals LR for fast L4 convergence
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE * 10,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,
)

print("✅ Model, MSELoss, Adam, and OneCycleLR scheduler initialised.")

---
## Cell 7 — Training Loop

**Task:** `"idr"` — ΔG thermodynamic stability regression on MegaScale cDNA.

With `ACCUMULATION_STEPS = 1` the optimiser steps every batch, giving the L4
a full native effective batch of 64.

In [ ]:
class _DataBatch:
    """Lightweight namespace — passes tensors through HoloGNN.forward."""
    __slots__ = ("input_ids", "mask", "edge_index")


def run_epoch(model, loader, criterion, optimizer, scheduler, device, train=True):
    """Single training or validation pass."""
    model.train() if train else model.eval()
    running_loss = 0.0
    if train: optimizer.zero_grad()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for i, batch in enumerate(tqdm(loader, leave=False)):
            data            = _DataBatch()
            data.input_ids  = batch["input_ids"].to(device, non_blocking=True)
            data.mask       = batch["attention_mask"].to(device, non_blocking=True)
            data.edge_index = None          # Triggers simple_linear_graph builder
            labels          = batch["label"].to(device, non_blocking=True)

            preds = model(data, task="idr")
            loss  = criterion(preds.squeeze(), labels)

            if train:
                (loss / ACCUMULATION_STEPS).backward()
                if (i + 1) % ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            running_loss += loss.item()

    # Flush any leftover gradients
    if train and len(loader) % ACCUMULATION_STEPS != 0:
        optimizer.step(); scheduler.step(); optimizer.zero_grad()

    return running_loss / len(loader)


# ── Main loop ────────────────────────────────────────────────────────────────
print("=" * 62)
print("  HOLO-GNN  ·  Full-Scale Training")
print(f"  Task          : IDR / ΔG Stability Regression")
print(f"  Device        : {device}")
print(f"  Batch size    : {BATCH_SIZE}  (accum × {ACCUMULATION_STEPS} = {BATCH_SIZE*ACCUMULATION_STEPS} eff.)")
print(f"  Epochs        : {EPOCHS}")
print("=" * 62)

history       = {"train": [], "val": []}
best_val      = float("inf")
wall_start    = time.time()

for epoch in range(1, EPOCHS + 1):
    t0         = time.time()
    train_loss = run_epoch(model, train_loader, criterion, optimizer, scheduler, device, train=True)
    val_loss   = run_epoch(model, val_loader,   criterion, optimizer, scheduler, device, train=False)
    elapsed    = time.time() - t0

    history["train"].append(train_loss)
    history["val"].append(val_loss)

    flag = ""
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        flag = "  ✅ saved"

    print(f"Epoch {epoch:>2}/{EPOCHS}  "
          f"train={train_loss:.4f}  val={val_loss:.4f}  "
          f"[{elapsed:.0f}s]{flag}")

total_hrs = (time.time() - wall_start) / 3600
print(f"\n🏁 Training complete in {total_hrs:.2f} h")
print(f"   Best val MSE : {best_val:.4f}")
print(f"   Checkpoint   : {CHECKPOINT_PATH}")

---
## Cell 8 — Loss Curve & Sanity Check

In [ ]:
import matplotlib.pyplot as plt

# ── Loss curve ────────────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-darkgrid")
fig, ax = plt.subplots(figsize=(8, 4))
ep = range(1, EPOCHS + 1)
ax.plot(ep, history["train"], marker="o", label="Train MSE")
ax.plot(ep, history["val"],   marker="s", label="Val MSE", linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.set_title("Holo-GNN  ·  ΔG Stability Regression — Training Curve")
ax.legend()
plt.tight_layout()
plt.savefig("training_loss_curve.png", dpi=150)
plt.show()
print("Loss curve → training_loss_curve.png")

# ── Inference sanity check ────────────────────────────────────────────────────
print("\nRunning sanity check on best checkpoint...")
model_eval = HoloGNN()
model_eval.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model_eval.to(device).eval()

sample = next(iter(val_loader))
with torch.no_grad():
    d            = _DataBatch()
    d.input_ids  = sample["input_ids"].to(device)
    d.mask       = sample["attention_mask"].to(device)
    d.edge_index = None
    preds = model_eval(d, task="idr").squeeze().cpu()

print(f"✅ Sanity check passed.")
print(f"   Predictions  (first 5): {preds[:5].tolist()}")
print(f"   True labels  (first 5): {sample['label'][:5].tolist()}")